In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("data/df_limpio.csv")
df

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Proposito,Fiador,Impago,Prima
0,6B4H6E,22,16000.0,10000,769,22.0,1,8.96,48.0,0.32,Escolar,Desempleado,Soltero,1.0,1.0,Vivienda,0.0,0,65.85
1,VKTUXP,33,38308.0,16474,699,70.0,3,14.07,36.0,0.20,Máster,Jornada completa,Casado,1.0,1.0,Vivienda,0.0,0,260.60
2,5P9OVL,51,51271.0,40770,812,309.0,1,12.66,12.0,0.36,Grado Universitario,Jornada completa,Casado,0.0,1.0,Vivienda,1.0,0,221.02
3,YL0W8G,30,31923.0,36707,813,43.0,1,8.96,24.0,0.32,Grado Universitario,Jornada completa,Casado,0.0,1.0,Vivienda,0.0,0,125.00
4,YP8WTV,28,27373.0,86981,765,60.0,2,10.07,48.0,0.55,Escolar,Jornada completa,Soltero,0.0,1.0,Vivienda,0.0,0,800.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101890,5PM51H,29,35671.0,5681,628,27.0,2,12.11,48.0,0.10,Grado Universitario,Jornada completa,Soltero,0.0,1.0,Vivienda,1.0,0,116.89
101891,L0RUJT,28,22864.0,40000,752,9.0,2,10.77,48.0,0.55,Grado Universitario,Jornada completa,Casado,0.0,1.0,Vivienda,0.0,0,607.55
101892,8OJZU0,35,69159.0,41795,699,65.0,2,10.41,36.0,0.29,Doctorado,Tiempo parcial,Soltero,0.0,0.0,Vivienda,0.0,0,406.82
101893,II8B24,39,68375.0,21847,722,170.0,2,6.26,36.0,0.14,Máster,Jornada completa,Casado,0.0,1.0,Vivienda,1.0,0,125.32


In [3]:
df.duplicated().sum()

np.int64(0)

No cambio ningún tipo de dato porque los que necesito ya están en numérico

Tenemos que saber el monto acordado hoy (monto_inicial), cuanto valdrá en el futuro (Cn)

In [4]:
#poner las variables que necesitamos en el dataframe
df["i"] = df["Ratio_Interes"] / 100   #interes en tanto por 1
df["n"] = df["Duracion"] / 12    # meses a años

In [5]:
# Cn = lo que valdrá el préstamo al final del plazo
df["Cn"] = df["Monto_Inicial"] * (1 + df["i"]) ** df["n"]  #despejando cn en la formula de C0

In [6]:
#valor actual (C0) con cada tipo de descuento
df["C0_racional_simple"]     = df["Cn"] * (1 + df["n"] * df["i"])**-1
df["C0_racional_compuesto"] = df["Cn"] * (1 + df["i"]) ** (-df["n"])
df["C0_comercial_simple"]    = df["Cn"] * (1 - df["n"] * df["i"])
df["C0_comercial_compuesto"] = df["Cn"] * (1 - df["i"]) ** df["n"]

In [7]:
# usamos el mismo tipo para i y para d
df["d"] = df["i"]

# préstamos donde el comercial simple falla (n*d es 1 o más)
invalidos = df[df["n"] * df["d"] >= 1]
print("Préstamos donde el comercial simple da C0 <= 0:", len(invalidos))

Préstamos donde el comercial simple da C0 <= 0: 136


In [8]:
# préstamos donde los cuatro métodos funcionan (n*d menor que 1)
validos = df[df["n"] * df["d"] < 1]

# porcentaje de lo prestado con cada tipo de descuento, solo con los válidos
total = validos["Monto_Inicial"].sum()
for tipo in ["racional_simple", "racional_compuesto", "comercial_simple", "comercial_compuesto"]:
    ratio = validos["C0_" + tipo].sum() / total * 100
    print(tipo, round(ratio, 1), "% de lo prestado")

racional_simple 103.6 % de lo prestado
racional_compuesto 100.0 % de lo prestado
comercial_simple 88.5 % de lo prestado
comercial_compuesto 96.1 % de lo prestado


In [9]:
#ver la cantidad descontada: D = Cn - C0
df["D_racional_simple"]     = df["Cn"] - df["C0_racional_simple"]
df["D_racional_compuesto"]  = df["Cn"] - df["C0_racional_compuesto"]
df["D_comercial_simple"]    = df["Cn"] - df["C0_comercial_simple"]
df["D_comercial_compuesto"] = df["Cn"] - df["C0_comercial_compuesto"]

In [10]:
#ver el resultado
df.head()

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,...,Cn,C0_racional_simple,C0_racional_compuesto,C0_comercial_simple,C0_comercial_compuesto,d,D_racional_simple,D_racional_compuesto,D_comercial_simple,D_comercial_compuesto
0,6B4H6E,22,16000.0,10000,769,22.0,1,8.96,48.0,0.32,...,14095.107039,10376.256654,10000.0,9043.420676,9682.720026,0.0896,3718.850385,4095.107039,5051.686363,4412.387013
1,VKTUXP,33,38308.0,16474,699,70.0,3,14.07,36.0,0.20,...,24451.943651,17194.250510,16474.0,14130.778236,15514.858594,0.1407,7257.693140,7977.943651,10321.165415,8937.085057
2,5P9OVL,51,51271.0,40770,812,309.0,1,12.66,12.0,0.36,...,45931.482000,40770.000000,40770.0,40116.556379,40116.556379,0.1266,5161.482000,5161.482000,5814.925621,5814.925621
3,YL0W8G,30,31923.0,36707,813,43.0,1,8.96,24.0,0.32,...,43579.584069,36956.906436,36707.0,35770.122604,36119.986478,0.0896,6622.677633,6872.584069,7809.461465,7459.597592
4,YP8WTV,28,27373.0,86981,765,60.0,2,10.07,48.0,0.55,...,127673.352448,91013.225298,86981.0,76246.526082,83506.183578,0.1007,36660.127150,40692.352448,51426.826366,44167.168870


In [11]:
df.columns

Index(['ID', 'Edad', 'Ingresos', 'Monto_Inicial', 'Scoring_Crediticio',
       'Meses_Empleo', 'Num_Creditos', 'Ratio_Interes', 'Duracion',
       'Ratio_Deuda_Ingresos', 'Estudios', 'Tipo_Jornada_Laboral',
       'Estado_Civil', 'Posesion_Hipoteca', 'Personas_Cargo', 'Proposito',
       'Fiador', 'Impago', 'Prima', 'i', 'n', 'Cn', 'C0_racional_simple',
       'C0_racional_compuesto', 'C0_comercial_simple',
       'C0_comercial_compuesto', 'd', 'D_racional_simple',
       'D_racional_compuesto', 'D_comercial_simple', 'D_comercial_compuesto'],
      dtype='object')

In [12]:
df.dtypes

ID                         object
Edad                        int64
Ingresos                  float64
Monto_Inicial               int64
Scoring_Crediticio          int64
Meses_Empleo              float64
Num_Creditos                int64
Ratio_Interes             float64
Duracion                  float64
Ratio_Deuda_Ingresos      float64
Estudios                   object
Tipo_Jornada_Laboral       object
Estado_Civil               object
Posesion_Hipoteca         float64
Personas_Cargo            float64
Proposito                  object
Fiador                    float64
Impago                      int64
Prima                     float64
i                         float64
n                         float64
Cn                        float64
C0_racional_simple        float64
C0_racional_compuesto     float64
C0_comercial_simple       float64
C0_comercial_compuesto    float64
d                         float64
D_racional_simple         float64
D_racional_compuesto      float64
D_comercial_si

In [13]:
df.isna().sum()

ID                        0
Edad                      0
Ingresos                  0
Monto_Inicial             0
Scoring_Crediticio        0
Meses_Empleo              0
Num_Creditos              0
Ratio_Interes             0
Duracion                  0
Ratio_Deuda_Ingresos      0
Estudios                  0
Tipo_Jornada_Laboral      0
Estado_Civil              0
Posesion_Hipoteca         0
Personas_Cargo            0
Proposito                 0
Fiador                    0
Impago                    0
Prima                     0
i                         0
n                         0
Cn                        0
C0_racional_simple        0
C0_racional_compuesto     0
C0_comercial_simple       0
C0_comercial_compuesto    0
d                         0
D_racional_simple         0
D_racional_compuesto      0
D_comercial_simple        0
D_comercial_compuesto     0
dtype: int64

In [14]:
total_prestado = df["Monto_Inicial"].sum()
for tipo in ["racional_simple", "racional_compuesto", "comercial_simple", "comercial_compuesto"]:
    ratio = df["C0_" + tipo].sum() / total_prestado * 100
    print(tipo, round(ratio, 1), "% de lo prestado")

racional_simple 103.7 % de lo prestado
racional_compuesto 100.0 % de lo prestado
comercial_simple 88.4 % de lo prestado
comercial_compuesto 96.1 % de lo prestado


- Hemos calculado cuánto vale hoy el dinero que se devolverá al final de cada préstamo con los cuatro tipos de descuento. El racional da siempre un valor mayor que el comercial. 
- El descuento comercial simplifica el tiempo, mientras que el racional es el adecuado para valorar y analizar ms a largo plazo, que es lo que necesita una aseguradora para valorar el riesgo.
- 

Para valorar el riesgo a valor presente es preferible el descuento racional frente al comercial: da siempre un valor actual mayor, es coherente con la capitalización y no falla en plazos largos. 

El descuento permite saber cuánto vale hoy el dinero que se cobrará o pagará al final de un préstamo.  El comercial simple es un 11,6 % menos y el comercial compuesto un 3,9 %, mientras que el racional simple se pasa con un 3,7 %. 

Se ha calculado cuánto vale hoy el dinero que se devolverá al final de cada préstamo con los cuatro tipos de descuento, usando el mismo tipo anual como i en el racional y como d en el comercial (supuesto). El racional da siempre un valor actual mayor que el comercial, porque aplica el tipo sobre lo que el dinero vale hoy y no directamente sobre el nominal.

El racional compuesto devuelve exactamente el monto inicial (100 %) porque Cn se obtuvo capitalizando de forma compuesta, así que es el único coherente con cómo se generó el dato. El racional simple se pasa un 3,7 % porque descuenta en simple algo que se capitalizó en compuesto. El comercial compuesto se queda un 3,9 % por debajo y el comercial simple un 11,6 % por debajo, y además este último no es válido cuando n·d ≥ 1 (136 préstamos de plazo largo y tipo alto, donde da cero o negativo). Por eso, para valorar el riesgo de una cartera a valor presente, el descuento racional compuesto es el más adecuad